# XTTS‑v2 Long‑Form TTS Pipeline (Single‑Shot, Long‑Narration Focus)

This notebook builds a robust long‑form speech synthesis workflow using Coqui XTTS‑v2 with a single‑shot inference target of ~5 minutes.

Key principles:
- Single‑shot synthesis first; avoid chunking unless absolutely necessary.
- Long‑form training samples to counter short‑clip bias.
- Text normalization to fix smart quotes, apostrophes, colons, and timing.
- One clean, short voice reference to lock identity.

Reference style line to prepend to your script:
"Speak clearly at a steady pace, small pauses at commas, longer pause at periods, friendly and professional tone."

Sections:
1. Setup (Python 3.11 check, installs)
2. Load XTTS‑v2
3. Text cleaning utilities
4. Data pull helpers (LibriTTS‑R subset, optional)
5. Optional LoRA fine‑tune (disabled by default)
6. Inputs: voice reference + trucking script (~800 words recommended)
7. Single‑shot inference
8. Two‑segment fallback with cross‑fade (only if needed)
9. Quality checks


In [13]:
import sys, subprocess, os
import platform

# Enforce Python 3.11 per project requirement
required_version = (3, 11)
if sys.version_info[:2] != required_version:
    raise RuntimeError(f"This notebook requires Python {required_version[0]}.{required_version[1]}")

# Install dependencies non-interactively
# 1) Pin torch/torchaudio to 2.5.1 to avoid torch 2.6 safe-unpickle changes
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-input", "--upgrade",
                "torch==2.5.1", "torchaudio==2.5.1"], check=True)

# 2) Install remaining deps with relaxed, compatible versions for XTTS‑v2 generate path (macOS arm64 friendly)
packages = [
    "TTS>=0.22.0,<0.23",
    "transformers>=4.39,<4.45",
    "accelerate>=0.33.0",
    "safetensors>=0.4.3",
    "sentencepiece>=0.2.0",
    "huggingface-hub>=0.24.0",
    "soundfile>=0.12.1",
    "librosa>=0.10.1",
    "datasets>=2.19.0"
]
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-input", "--upgrade"] + packages, check=True)
except subprocess.CalledProcessError:
    # Fallback: retry without version pins for HF deps if initial install fails
    fallback = [
        "TTS",
        "transformers",
        "accelerate",
        "safetensors",
        "sentencepiece",
        "huggingface-hub",
        "soundfile",
        "librosa",
        "datasets"
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-input", "--upgrade"] + fallback, check=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch as _torch
    import transformers as _tf
    import TTS as _coqui
    print("Torch:", _torch.__version__)
    print("Transformers:", _tf.__version__)
    print("Coqui TTS:", _coqui.__version__)
except Exception:
    pass
print("CUDA/MPS availability auto-detected by torch if present.")


Python: 3.11.13 (main, Jun  3 2025, 18:38:25) [Clang 17.0.0 (clang-1700.0.13.3)]
Platform: macOS-15.6.1-arm64-arm-64bit
Torch: 2.5.1
Transformers: 4.44.2
Coqui TTS: 0.22.0
CUDA/MPS availability auto-detected by torch if present.


In [14]:
import os

# Accept Coqui CPML non-interactively for XTTS‑v2 usage (non-commercial by default)
os.environ["COQUI_TOS_AGREED"] = "1"

# Enable CPU fallback for unsupported Apple MPS ops (fixes conv1d NotImplementedError)
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Disable MPS selection entirely to avoid XTTS speaker encoder conv1d limitation
try:
    import torch
    if hasattr(torch.backends, "mps"):
        torch.backends.mps.is_available = lambda: False  # force device selection to skip MPS
        print("MPS disabled via runtime monkeypatch")
except Exception as _e:
    print("MPS disable skipped:", _e)

print("Set COQUI_TOS_AGREED=1 and PYTORCH_ENABLE_MPS_FALLBACK=1")


MPS disabled via runtime monkeypatch
Set COQUI_TOS_AGREED=1 and PYTORCH_ENABLE_MPS_FALLBACK=1


In [15]:
# Register PyTorch safe globals for XTTS configs when using torch>=2.6 defaults
import torch.serialization

try:
    torch.serialization.add_safe_globals(["TTS.tts.configs.xtts_config.XttsConfig"])  # type: ignore[attr-defined]
    print("Registered torch safe_globals for XttsConfig")
except Exception as e:
    print("safe_globals registration skipped:", e)


Registered torch safe_globals for XttsConfig


In [16]:
# Force XTTS to use Hugging Face generate path if supported
try:
    from TTS.tts.configs.xtts_config import XttsConfig
    cfg = XttsConfig()
    cfg.use_hf_generate = True  # prefer HF generate over internal inference if available
    print("Configured use_hf_generate=True")
except Exception as e:
    print("use_hf_generate not set:", e)


Configured use_hf_generate=True


In [17]:
from TTS.api import TTS
import torch

# Load XTTS‑v2; prefer GPU, then Apple MPS, else CPU
if torch.cuda.is_available():
    _device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    _device = "mps"
else:
    _device = "cpu"

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(_device)
print("Loaded XTTS‑v2 on", _device)


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


/opt/homebrew/lib/python3.11/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.speakers = torch.load(speaker_file_path)

Loaded XTTS‑v2 on cpu


In [18]:
import re

SMART_QUOTES = {
    "“": '"', "”": '"',
    "‘": "'", "’": "'",
}

CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "n't": " not",
    "it's": "it is", "I'm": "I am", "we're": "we are", "we'll": "we will",
    "they're": "they are", "they'll": "they will", "you're": "you are",
    "I've": "I have", "we've": "we have", "they've": "they have",
}

# User preference: remove emojis/symbols entirely; keep letters, digits, punctuation we permit (including em dash)
_ALLOWED_CHARS = r"A-Za-z0-9 ,\.\-\?\!;:'\"\(\)\n\r\t—"

def clean_text(text: str) -> str:
    # Convert smart quotes
    for s, p in SMART_QUOTES.items():
        text = text.replace(s, p)
    # Expand contractions (case-insensitive where possible)
    for c, e in CONTRACTIONS.items():
        text = re.sub(rf"\b{re.escape(c)}\b", e, text, flags=re.IGNORECASE)
    # Colon handling: replace bare ':' with ': ' if missing space; allow em dash with spaces
    text = re.sub(r"\s*:\s*", ": ", text)
    text = re.sub(r"\s*—\s*", " — ", text)
    # Add gentle prosody hints around sentence punctuation
    text = re.sub(r"\s*([,])\s*", r", ", text)
    text = re.sub(r"\s*([\.\!\?])\s*", r"\1 ", text)
    # Remove disallowed symbols (including emojis)
    text = re.sub(rf"[^{_ALLOWED_CHARS}]", "", text)
    # Collapse extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Quick check
example = '“It’s time: follow—now.” We’ll go.'
print(clean_text(example))


"it is time: follow — now. " we will go.


In [19]:
from datasets import load_dataset
from pathlib import Path
import librosa

# Optional: small long‑form subset helper from LibriTTS‑R
# Note: LibriTTS‑R HF identifier can vary; fallback to LibriTTS if unavailable.
# Try common identifiers in order; if not present, skip quietly.

def try_load_librir_like(split: str = "train"):
    candidates = [
        # Known LibriTTS on HF; LibriTTS‑R may be mirrored under similar names.
        "patrickvonplaten/librispeech_asr_dummy",  # tiny for sanity only
        "lj_speech",  # single‑speaker fallback for code path checks
    ]
    ds = None
    for name in candidates:
        try:
            ds = load_dataset(name, split=split)
            print(f"Loaded dataset: {name} ({split})")
            break
        except Exception:
            continue
    return ds

# Filter for longer utterances by duration (20–40s) where possible

def select_long_samples(ds, audio_column: str = None, min_s: float = 20.0, max_s: float = 40.0, limit: int = 50):
    if ds is None:
        return []
    samples = []
    # Best effort: if dataset exposes 'audio' dict with 'array'/'path'
    for row in ds:
        wav_path = None
        if isinstance(row.get("audio"), dict) and row["audio"].get("path"):
            wav_path = row["audio"]["path"]
        elif audio_column and row.get(audio_column):
            wav_path = row[audio_column]
        if not wav_path or not Path(wav_path).exists():
            continue
        try:
            dur = librosa.get_duration(filename=wav_path)
            if min_s <= dur <= max_s:
                samples.append({"path": wav_path, "duration": dur, "text": row.get("text") or row.get("normalized_text")})
                if len(samples) >= limit:
                    break
        except Exception:
            continue
    print(f"Selected {len(samples)} long samples")
    return samples

_ds = try_load_librir_like()
_long_samples = select_long_samples(_ds)
len(_long_samples)


0

In [20]:
import numpy as np
import soundfile as sf

# Cross‑fade join helper for the two‑segment fallback (pure NumPy/SoundFile)

def cross_fade_join(wav1_path: str, wav2_path: str, out_path: str, fade_ms: int = 250):
    y1, sr1 = sf.read(wav1_path, always_2d=True)
    y2, sr2 = sf.read(wav2_path, always_2d=True)
    if sr1 != sr2:
        raise ValueError("Sample rates do not match")
    fade_samples = max(1, int(sr1 * (fade_ms / 1000.0)))
    if y1.shape[0] <= fade_samples or y2.shape[0] <= fade_samples:
        # If too short to crossfade, just butt-join
        y = np.concatenate([y1, y2], axis=0)
        sf.write(out_path, y, sr1)
        return out_path
    a = y1[:-fade_samples]
    b = y2[fade_samples:]
    # Linear fade windows
    fade_out = np.linspace(1.0, 0.0, fade_samples, dtype=np.float32)[:, None]
    fade_in = 1.0 - fade_out
    overlap = y1[-fade_samples:] * fade_out + y2[:fade_samples] * fade_in
    y = np.concatenate([a, overlap, b], axis=0)
    sf.write(out_path, y, sr1)
    return out_path

print("cross_fade_join ready (NumPy/SoundFile)")


cross_fade_join ready (NumPy/SoundFile)


In [21]:
# Inputs: set your voice reference WAV and the trucking agent script here

REFERENCE_WAV = "./reference.wav"  # TODO: place a 6–15s clean mono 22.05–24 kHz clip

STYLE_LINE = (
    "Speak clearly at a steady pace, small pauses at commas, longer pause at periods, "
    "friendly and professional tone."
)

TRUCKING_SCRIPT = (
   ''' Hello. This is your dispatch assistant. I am here to help you with pickup and delivery details, directions, timing, paperwork, and anything else that comes up on the road. If you need me to contact someone, say the word and I will draft the message or place the call through dispatch.

First, let us confirm the route. Tell me your current city, your trailer number, and the load reference or pickup number. I will look up the shipper or receiver address, the planned route, and the target appointment time. If you already see a posted ETA on your ELD, tell me what it shows. If not, I will calculate an ETA for you.

If you need directions, I can read them step by step. Say, give me directions from here to the shipper, or from the shipper to the receiver. I will call out major highways, safe truck exits, and the last mile note. If there are truck restrictions such as low bridges, weight limits, or no truck streets, I will mention them. When you are close, I can switch to short turn by turn instructions.

Before you leave, let us check your pickup details. Do you have the pickup number, the bill of lading number if it is already assigned, and the shipper phone? If not, I can request that now. If the shipper is a large facility, I can tell you the gate to use, whether they require a safety vest, and if a check in kiosk is on site. If you need a pickup appointment window, I will ask dispatch to confirm the time.

On the way to the shipper, I can watch weather and traffic along your route. If heavy weather or a major delay appears ahead, I will reroute you if a safe alternate exists, or I will notify dispatch and update the ETA. If you want a rest break or fuel, say plan a stop, and I will suggest a safe option based on truck parking, fuel network, and remaining drive time.

At the shipper, if you need check in steps, ask me to read them. I can remind you to record the arrival time for detention, note the trailer and seal requirements, and check whether a load is live or drop and hook. If it is a live load, I will track the time on the dock and can start a detention timer if we have that agreement. If it is drop and hook, I will list the yard rules such as chock the wheels, drop gear low, and use a red tag if required.

If you are hauling refrigerated freight, say reefer help. I will confirm set point, continuous or cycle, pre cool time, and fuel level. If a temp recorder or pulp temperature is required, I will remind you to get those numbers on the bill. If the unit alarms, tell me the code and I will read the quick fix steps or call the service line.

Once you get the bills, check the pallet count, weight, and any notations. If the load is short or there is an overage or damage, say create an OS and D note, and I will record details and notify dispatch. If you need pictures for claims, I will tell you exactly what to capture. Remember to confirm that the seal number on the bills matches the seal on the trailer.

When you roll toward the receiver, I will keep ETA updated and share yard notes. If there is a delivery appointment window, I will monitor it. If you will be early and they allow early arrivals, I can request approval. If you will be late because of traffic, weather, or a breakdown, say I am delayed and tell me how long. I will alert dispatch, update the receiver, and adjust the plan.

If you need to contact the receiver or shipper directly, say call the receiver or call the shipper. I will draft a clear message with your name, trailer, reference number, and status. If dispatch needs to speak with them, I will hand it off and confirm the result.

For safety checks, if you want a quick reminder, say safety checklist. I will run through tires, lights, brakes, glad hands, fifth wheel, doors, seal, load locks, and if you are in winter, chains and washer fluid. If it is a hot day and you have a reefer, we will verify airflow, drain holes, and door gaskets. If you are carrying hazmat, I will confirm placards and shipping papers.

At the receiver, if there is a guard shack, I can read the check in steps. If there is a lumping service, I will remind you to request a receipt. If it is a live unload, I will track time on dock for detention. If the freight is count by driver, I will go through the count instructions. If there is a shortage or damage, we will log a note on the proof of delivery and notify dispatch.

When you are empty, say I am empty, and I will request the next move. If there is a backhaul or another pickup, I will give you the new address, pickup number, and timing. If you need directions, fuel, or a nearby safe place to park before the next load, I will plan it for you.

If you have a breakdown, say breakdown help. Tell me what happened and where you are. I will record your location, mile marker if you have it, and the issue. I will contact roadside or the maintenance provider and share the ETA. I can also message dispatch and the shipper or receiver if this affects a pickup or delivery time.

If there is an incident or accident, say incident report. I will help you capture the basics such as time, location, photos, other parties, and a brief description. Then I will notify safety and dispatch and wait for next steps.

For long routes, if you want a plan, say build a plan. I will set a target pace, rest breaks, fuel stops, and overnight parking based on your hours and distance. If scales or inspections appear on your route, I will remind you to keep paperwork handy.

If you need to double check anything, ask me. If a number looks wrong, I will re read it. If you want me to slow down, I will pause more between sentences. If you want me to speed up, I will keep answers short and direct.

Here is a quick summary of what I can do for you. Confirm addresses, pickup numbers, and appointment windows. Give directions and last mile notes. Track weather, traffic, and ETAs. Handle delay messages and detention times. Manage reefer settings and alarms. Log OS and D with pictures and notes. Guide check in and check out steps. Help with proof of delivery and lumper receipts. Plan fuel and rest stops. Call or message a shipper, a receiver, or dispatch when you ask. And, if something goes wrong, I will escalate it fast.

If you want to start now, tell me your current city, your trailer number, and the next reference number. I will pull up the stop and get you moving.'''
)

print("Reference:", REFERENCE_WAV)
print("Script words:", len(TRUCKING_SCRIPT.split()))


Reference: ./reference.wav
Script words: 1202


In [22]:
# Single‑shot inference first
from pathlib import Path
from datetime import datetime

CLEANED_TEXT = clean_text(TRUCKING_SCRIPT)
print("Cleaned words:", len(CLEANED_TEXT.split()))

out_single = f"./xtts_single.wav"

# Generate in one go
# Note: XTTS‑v2 expects a path to reference WAV; ensure file exists and is mono 22.05–24 kHz.
if not Path(REFERENCE_WAV).exists():
    raise FileNotFoundError(f"Missing REFERENCE_WAV: {REFERENCE_WAV}")

# language 'en' for English
_ = tts.tts_to_file(text=CLEANED_TEXT, speaker_wav=REFERENCE_WAV, language="en", file_path=out_single)
print("Wrote:", out_single)


Cleaned words: 1202
 > Text splitted to sentences.
['Hello.', 'This is your dispatch assistant.', 'I am here to help you with pickup and delivery details, directions, timing, paperwork, and anything else that comes up on the road.', 'If you need me to contact someone, say the word and I will draft the message or place the call through dispatch.', 'First, let us confirm the route.', 'Tell me your current city, your trailer number, and the load reference or pickup number.', 'I will look up the shipper or receiver address, the planned route, and the target appointment time.', 'If you already see a posted ETA on your ELD, tell me what it shows.', 'If not, I will calculate an ETA for you.', 'If you need directions, I can read them step by step.', 'Say, give me directions from here to the shipper, or from the shipper to the receiver.', 'I will call out major highways, safe truck exits, and the last mile note.', 'If there are truck restrictions such as low bridges, weight limits, or no truck 

In [23]:
# Two‑segment fallback if OOM or model limit occurs
from math import ceil

SEGMENT_BREAK = 0.6  # rough fraction for first segment (e.g., ~3 minutes then ~2)

# Simple split by words, keeping sentence boundaries where possible

def split_text_for_two_segments(text: str):
    words = text.split()
    if len(words) < 50:
        return [text]
    cut = int(len(words) * SEGMENT_BREAK)
    # try to extend cut to next period/comma within a window
    window = min(len(words) - cut - 1, 80)
    for i in range(window):
        token = words[cut + i]
        if token.endswith('.') or token.endswith('!') or token.endswith('?'):
            cut = cut + i + 1
            break
    return [' '.join(words[:cut]), ' '.join(words[cut:])]

parts = split_text_for_two_segments(CLEANED_TEXT)
print("Segments:", [len(p.split()) for p in parts])

if len(parts) == 2:
    out_a = f"./xtts_segA.wav"
    out_b = f"./xtts_segB.wav"
    _ = tts.tts_to_file(text=parts[0], speaker_wav=REFERENCE_WAV, language="en", file_path=out_a)
    _ = tts.tts_to_file(text=parts[1], speaker_wav=REFERENCE_WAV, language="en", file_path=out_b)
    out_joined = f"./xtts_joined.wav"
    cross_fade_join(out_a, out_b, out_joined, fade_ms=300)
    print("Wrote joined:", out_joined)
else:
    print("Two‑segment fallback not used (single‑shot applied)")


Segments: [723, 479]
 > Text splitted to sentences.
['Hello.', 'This is your dispatch assistant.', 'I am here to help you with pickup and delivery details, directions, timing, paperwork, and anything else that comes up on the road.', 'If you need me to contact someone, say the word and I will draft the message or place the call through dispatch.', 'First, let us confirm the route.', 'Tell me your current city, your trailer number, and the load reference or pickup number.', 'I will look up the shipper or receiver address, the planned route, and the target appointment time.', 'If you already see a posted ETA on your ELD, tell me what it shows.', 'If not, I will calculate an ETA for you.', 'If you need directions, I can read them step by step.', 'Say, give me directions from here to the shipper, or from the shipper to the receiver.', 'I will call out major highways, safe truck exits, and the last mile note.', 'If there are truck restrictions such as low bridges, weight limits, or no truck

In [24]:
# Quality checks: duration, silence heuristic, and quick listen stats
import soundfile as sf
import numpy as np

last_output = None
for cand in [f"./xtts_joined.wav", out_single]:
    if Path(cand).exists():
        last_output = cand
        break

if last_output is None:
    raise FileNotFoundError("No synthesized output found.")

wav, sr = sf.read(last_output)
duration_sec = len(wav) / sr

# Simple silence heuristic: fraction of samples with very low amplitude
abs_wav = np.abs(wav if wav.ndim == 1 else wav.mean(axis=1))
silence_fraction = float((abs_wav < 1e-4).mean())

print({
    "file": last_output,
    "sr": sr,
    "duration_sec": round(duration_sec, 2),
    "silence_fraction": round(silence_fraction, 4),
})


{'file': './xtts_joined.wav', 'sr': 24000, 'duration_sec': 403.13, 'silence_fraction': 0.1206}


## Optional: Quick LoRA fine‑tune block (disabled by default)

This section sketches a minimal LoRA fine‑tune approach focused on paragraph‑length samples. It is provided as a placeholder; consult Coqui docs for production training.

- Curate 200–400 paragraphs, each 20–40 seconds.
- Use cross‑entropy loss with stable defaults; do not invent custom losses.
- Keep the same reference voice at inference to lock identity.

To enable, implement a proper dataset and training runner with checkpointing, then point inference to the fine‑tuned checkpoint.
